In [1]:
from langchain.tools import tool
import osmnx as ox

@tool
def query_osm(address : str, tags : dict, dist : int) -> list[dict]:
    """
    Запрашивает объекты из OpenStreetMap в радиусе dist метров от адреса address, которые соответствуют тегам tags.
    Возвращает список словарей, каждый из которых содержит информацию об объекте.
    """
    try:
        features = ox.features_from_address(address, tags=tags, dist=dist).drop(columns=['geometry'])
        results = [
            f[~f.isna()].to_dict()
            for _,f in features.iterrows()
        ]
        return results
    except Exception as e:
        return [{"error": str(e)}]

In [ ]:
from urbanomy.methods.agent import Agent

agent = Agent(system_prompt="Ты эксперт по урбанистике. Используй инструменты для доступа к данным из OSM. Не придумывай информацию, основывайся только на результатах вызова инструментов.", tools=[query_osm], debug=True)

print(agent.invoke("""
Дан следующий адрес:
Санкт-Петербург, Белградская улица 28к1
Есть ли в радиусе 3 километров больница? 
"""))

[values] {'messages': [HumanMessage(content='\nДан следующий адрес:\nСанкт-Петербург, Белградская улица 28к1\nЕсть ли в радиусе 3 километров больница? \n', additional_kwargs={}, response_metadata={}, id='18acf029-7990-4b16-b468-9bca3a94dbf5')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 250, 'total_tokens': 363, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gpt-oss:120b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-176', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6396-e565-7cc1-bd1a-9657a8f88c42-0', tool_calls=[{'name': 'query_osm', 'args': {'address': 'Санкт-Петербург, Белградская улица 28к1', 'dist': 3000, 'tags': {'amenity': 'hospital'}}, 'id': 'call_2iyxf8em', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 250, 'output_tokens': 113, 